In [1]:
from pyspark.sql import SparkSession, functions as F, types as T

In [2]:
spark = SparkSession.builder.master('local[*]').getOrCreate()

In [3]:
print(spark.version)

3.5.1


In [4]:
actor_df = spark.read.csv('./data/actor.csv', header=True, inferSchema=True)
address_df = spark.read.csv('./data/address.csv', header=True, inferSchema=True)
category_df = spark.read.csv('./data/category.csv', header=True, inferSchema=True)
city_df = spark.read.csv('./data/city.csv', header=True, inferSchema=True)
country_df = spark.read.csv('./data/country.csv', header=True, inferSchema=True)
customer_df = spark.read.csv('./data/customer.csv', header=True, inferSchema=True)
film_df = spark.read.csv('./data/film.csv', header=True, inferSchema=True)
film_actor_df = spark.read.csv('./data/film_actor.csv', header=True, inferSchema=True)
film_category_df = spark.read.csv('./data/film_category.csv', header=True, inferSchema=True)
inventory_df = spark.read.csv('./data/inventory.csv', header=True, inferSchema=True)
language_df = spark.read.csv('./data/language.csv', header=True, inferSchema=True)
payment_df = spark.read.csv('./data/payment.csv', header=True, inferSchema=True)
rental_df = spark.read.csv('./data/rental.csv', header=True, inferSchema=True)
staff_df = spark.read.csv('./data/staff.csv', header=True, inferSchema=True)
store_df = spark.read.csv('./data/store.csv', header=True, inferSchema=True)

# Домашнє завдання на тему Spark SQL

Задачі з домашнього завдання на SQL потрібно розвʼязати за допомогою Spark SQL DataFrame API.

- Дампи таблиць знаходяться в папці `data`. Датафрейми таблиць вже створені в клітинці вище.
- Можете створювати стільки нових клітинок, скільки вам необхідно.
- Розвʼязок кожної задачі має бути відображений в самому файлі (використати метод `.show()`)
- код має бути оформлений у відповідності із одним із стилем, показаним лектором на занятті 13.

**Увага!**
Використовувати мову запитів SQL безпосередньо забороняється, потрібно використовувати виключно DataFrame API!


1.
Вивести кількість фільмів в кожній категорії.
Результат відсортувати за спаданням.

In [7]:
categories = film_category_df.join(category_df, on="category_id", how="inner")
result = categories.groupBy("name").agg(F.count("film_id").alias("film_count")).orderBy(F.col("film_count").desc())

print("Кількість фільмів у кожній категорії:")
result.select("name", "film_count").show()

Кількість фільмів у кожній категорії:
+-----------+----------+
|       name|film_count|
+-----------+----------+
|     Sports|        74|
|    Foreign|        73|
|     Family|        69|
|Documentary|        68|
|  Animation|        66|
|     Action|        64|
|        New|        63|
|      Drama|        62|
|      Games|        61|
|     Sci-Fi|        61|
|   Children|        60|
|     Comedy|        58|
|     Travel|        57|
|   Classics|        57|
|     Horror|        56|
|      Music|        51|
+-----------+----------+



2.
Вивести 10 акторів, чиї фільми брали на прокат найбільше.
Результат відсортувати за спаданням.

In [9]:
films_cnt_rentals = rental_df.join(inventory_df, on="inventory_id", how="inner").select("rental_id", "film_id") \
                    .groupBy(F.col("film_id")).agg(F.countDistinct("rental_id").alias("cnt_rentals"))
actors = films_cnt_rentals.join(film_actor_df, on="film_id", how="inner").select("film_id", "actor_id", "cnt_rentals")
actor_names = actors.join(actor_df, on="actor_id", how="inner").select("cnt_rentals", "actor_id", "first_name", "last_name")

print("ТОП 10 акторів, чиї фільми брали на прокат найбільше:")
# actor_names.orderBy(F.col("cnt_rentals").desc()).show(10)
actor_names.orderBy(F.col("cnt_rentals").desc()).limit(10).show()

ТОП 10 акторів, чиї фільми брали на прокат найбільше:
+-----------+--------+----------+-----------+
|cnt_rentals|actor_id|first_name|  last_name|
+-----------+--------+----------+-----------+
|         34|      32|       TIM|    HACKMAN|
|         34|      51|      GARY|    PHOENIX|
|         34|     193|      BURT|     TEMPLE|
|         34|      92|   KIRSTEN|     AKROYD|
|         34|      89|  CHARLIZE|      DENCH|
|         34|      26|       RIP|   CRAWFORD|
|         33|      42|       TOM|    MIRANDA|
|         33|     194|     MERYL|      ALLEN|
|         33|      35|      JUDY|       DEAN|
|         33|     195|     JAYNE|SILVERSTONE|
+-----------+--------+----------+-----------+



3.
Вивести категорія фільмів, на яку було витрачено найбільше грошей
в прокаті

In [11]:
rental_films = rental_df.join(inventory_df, on="inventory_id", how="inner").select("rental_id", "film_id")
categories = rental_films.join(film_category_df, on="film_id", how="inner").select("rental_id", "film_id", "category_id")
amount_by_categories = categories.join(payment_df, on="rental_id", how="inner").select("rental_id", "film_id", "category_id", "amount") \
                      .groupBy(F.col("category_id")).agg(F.sum("amount").alias("amount_rental"))
result = amount_by_categories.join(category_df, on="category_id", how="inner").select("category_id", "name", "amount_rental")

print("Категорія фільмів, на яку було витрачено найбільше грошей в прокаті:")
result.orderBy(F.col("amount_rental").desc()).limit(1).show()

Категорія фільмів, на яку було витрачено найбільше грошей в прокаті:
+-----------+------+-----------------+
|category_id|  name|    amount_rental|
+-----------+------+-----------------+
|         15|Sports|5314.209999999848|
+-----------+------+-----------------+



4.
Вивести назви фільмів, яких не має в inventory.

In [13]:
result = film_df.join(inventory_df, on="film_id", how="left_anti").select("title")

print("Назви фільмів, яких не має в inventory:")
result.show()

Назви фільмів, яких не має в inventory:
+--------------------+
|               title|
+--------------------+
|      ALICE FANTASIA|
|         APOLLO TEEN|
|      ARGONAUTS TOWN|
|       ARK RIDGEMONT|
|ARSENIC INDEPENDENCE|
|   BOONDOCK BALLROOM|
|       BUTCH PANTHER|
|       CATCH AMISTAD|
| CHINATOWN GLADIATOR|
|      CHOCOLATE DUCK|
|COMMANDMENTS EXPRESS|
|    CROSSING DIVORCE|
|     CROWDS TELEMARK|
|    CRYSTAL BREAKING|
|          DAZED PUNK|
|DELIVERANCE MULHO...|
|   FIREHOUSE VIETNAM|
|       FLOATS GARDEN|
|FRANKENSTEIN STRA...|
|  GLADIATOR WESTWARD|
+--------------------+
only showing top 20 rows



5.
Вивести топ 3 актори, які найбільше зʼявлялись в категорії фільмів “Children”

In [15]:
category_children = category_df.filter(F.col("name")=="Children").first()["category_id"]
films_of_child_category = film_category_df.filter(F.col("category_id")==category_children).select("film_id")
actors_of_child_category = films_of_child_category.join(film_actor_df, on="film_id", how="inner").select("actor_id")
actor_names = actors_of_child_category.join(actor_df, on="actor_id", how="inner")
films_of_actors = actor_names.groupBy(F.col("first_name"), F.col("last_name")).agg(F.count("actor_id").alias("cnt_films_of_actor"))

print("ТОП 3 актори, які найбільше зʼявлялись в категорії фільмів “Children”:")
films_of_actors.orderBy(F.col("cnt_films_of_actor").desc()).limit(3).show()

ТОП 3 актори, які найбільше зʼявлялись в категорії фільмів “Children”:
+----------+---------+------------------+
|first_name|last_name|cnt_films_of_actor|
+----------+---------+------------------+
|     HELEN|   VOIGHT|                 7|
|     SUSAN|    DAVIS|                 6|
|     RALPH|     CRUZ|                 5|
+----------+---------+------------------+



Stop Spark session:

In [17]:
spark.stop()